The Model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import csv
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


In [1]:
# HELPER FUNCTIONS

def read_glove_vecs(glove_file):
    with open(glove_file, 'r') as f:
        words = set()
        word_to_vec_map = {}
        for line in f:
            line = line.strip().split()
            curr_word = line[0]
            words.add(curr_word)
            word_to_vec_map[curr_word] = np.array(line[1:], dtype=np.float64)
        
        i = 1
        words_to_index = {}
        index_to_words = {}
        for w in sorted(words):
            words_to_index[w] = i
            index_to_words[i] = w
            i = i + 1
    return words_to_index, index_to_words, word_to_vec_map

def convert_to_one_hot(Y, C):
    Y = np.eye(C)[Y.reshape(-1)]
    return Y

def read_csv(filename):
    phrase = []
    emoji = []

    with open (filename) as csvDataFile:
        csvReader = csv.reader(csvDataFile)

        for row in csvReader:
            phrase.append(row[0])
            emoji.append(row[1])

    X = np.asarray(phrase)
    Y = np.asarray(emoji, dtype=int)

    return X, Y

In [ ]:
X_train, Y_train = read_csv('data/train.csv')
X_test, Y_test = read_csv('data/test.csv')


In [ ]:
Y_oh_train = convert_to_one_hot(Y_train, C = 5)
Y_oh_test = convert_to_one_hot(Y_test, C = 5)

In [ ]:
word_to_index, index_to_word, word_to_vec_map = read_glove_vecs('data/glove.6B.50d.txt')


In [ ]:
def sentences_to_indices(X, word_to_index, max_len):
    """
    Converts an array of sentences (strings) into an array of indices corresponding to words in the sentences.
    """
    
    m = X.shape[0]  # number of training examples
    
    # Initialize X_indices as a numpy matrix of zeros and the correct shape
    X_indices = np.zeros((m,max_len))
    
    for i in range(m):  # loop over training examples
        
        # Convert the ith sentence in lower case and split into a list of words
        sentence_words = X[i].lower().split()
        
        # Initialize j to 0
        j = 0
        
        # Loop over the words of sentence_words
        for w in sentence_words:
            # Set the (i,j)th entry of X_indices to the index of the correct word.
            X_indices[i, j] = word_to_index[w]
            # Increment j to j + 1
            j = j + 1
    
    return X_indices

In [ ]:
X1 = np.array(["lol", "I love you", "this is very yummy"])
X1_indices = sentences_to_indices(X1,word_to_index, max_len = 5)
print("X1 =", X1)
print("X1_indices =", X1_indices)

Defining the Network using Pretrained Embedding Layer using GloVe Word Embeddings

In [ ]:
class NN(nn.Module):
    def __init__(self, embedding, embedding_dim, hidden_dim, vocab_size, output_dim, batch_size):
        super(NN, self).__init__()
        self.batch_size = batch_size
        self.hidden_dim = hidden_dim
        self.word_embeddings = embedding
        self.lstm = nn.LSTM(embedding_dim,
                            hidden_dim,
                            num_layers=2,
                            dropout=0.5,
                            batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, sentence):
        sentence = sentence.to(device)
        embeds = self.word_embeddings(sentence)
        h0 = torch.zeros(2, sentence.size(0), self.hidden_dim).requires_grad_().to(device)
        c0 = torch.zeros(2, sentence.size(0), self.hidden_dim).requires_grad_().to(device)
        lstm_out, _ = self.lstm(embeds, (h0, c0))
        lstm_out = lstm_out[:, -1, :]
        lstm_out = F.dropout(lstm_out, 0.5)
        out = self.fc(lstm_out)
        out = F.softmax(out, dim=1)
        return out


Creating the Glove Embedding Layer

In [ ]:
def pretrained_embedding_layer(word_to_vec_map, word_to_index, non_trainable=True):
    num_embeddings = len(word_to_index) + 1                   
    embedding_dim = word_to_vec_map["cucumber"].shape[0]  #  dimensionality of GloVe word vectors (= 50)

    # Initialize the embedding matrix as a numpy array of zeros of shape (num_embeddings, embedding_dim)
    weights_matrix = np.zeros((num_embeddings, embedding_dim))

    # Set each row "index" of the embedding matrix to be the word vector representation of the "index"th word of the vocabulary
    for word, index in word_to_index.items():
        weights_matrix[index, :] = word_to_vec_map[word]

    embed = nn.Embedding.from_pretrained(torch.from_numpy(weights_matrix).type(torch.FloatTensor), freeze=non_trainable)

    return embed, num_embeddings, embedding_dim

Training the model

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

def train(model, train_loader, test_loader, criterion, optimizer, epochs=10):
    model.to(device)
    train_losses, test_losses, accuracies = [], [], []

    for e in range(epochs):
        running_loss = 0
        model.train()

        for sentences, labels in train_loader:
            sentences, labels = sentences.to(device), labels.to(device)
            optimizer.zero_grad()
            pred = model(sentences)
            loss = criterion(pred, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        model.eval()
        test_loss = 0
        accuracy = 0
        with torch.no_grad():
            for sentences, labels in test_loader:
                sentences, labels = sentences.to(device), labels.to(device)
                log_ps = model(sentences)
                test_loss += criterion(log_ps, labels)
                top_p, top_class = log_ps.topk(1, dim=1)
                equals = top_class == labels.view(*top_class.shape)
                accuracy += torch.mean(equals.type(torch.FloatTensor))

        train_losses.append(running_loss / len(train_loader))
        test_losses.append(test_loss / len(test_loader))
        accuracies.append(accuracy / len(test_loader) * 100)

        print('Epoch: {}/{}..'.format(e + 1, epochs),
              'Training Loss: {:.3f}..'.format(running_loss / len(train_loader)),
              'Test Loss: {:.3f}..'.format(test_loss / len(test_loader)),
              'Test Accuracy: {:.3f}'.format(accuracy / len(test_loader)))

    plt.figure(figsize=(20, 5))
    plt.plot(train_losses, c='b', label='Training loss')
    plt.plot(test_losses, c='r', label='Testing loss')
    plt.xticks(np.arange(0, epochs))
    plt.title('Losses')
    plt.legend(loc='upper right')
    plt.show()

    plt.figure(figsize=(20, 5))
    plt.plot(accuracies)
    plt.xticks(np.arange(0, epochs))
    plt.title('Accuracy')
    plt.show()


In [ ]:
import torch.utils.data

maxLen = len(max(X_train, key=len).split())
X_train_indices = sentences_to_indices(X_train, word_to_index, maxLen)
X_test_indices = sentences_to_indices(X_test, word_to_index, maxLen)

embedding, vocab_size, embedding_dim = pretrained_embedding_layer(word_to_vec_map, word_to_index, non_trainable=True)

hidden_dim = 128
output_size = 5
batch_size = 32

model = NN(embedding, embedding_dim, hidden_dim, vocab_size, output_size, batch_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)
epochs = 50

train_dataset = torch.utils.data.TensorDataset(
    torch.tensor(X_train_indices).type(torch.LongTensor),
    torch.tensor(Y_train).type(torch.LongTensor)
)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size)

test_dataset = torch.utils.data.TensorDataset(
    torch.tensor(X_test_indices).type(torch.LongTensor),
    torch.tensor(Y_test).type(torch.LongTensor)
)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size)

train(model, train_loader, test_loader, criterion, optimizer, epochs)


### Training Output (50 epochs)

```
Epoch: 1/50..  Training Loss: 1.602..  Test Loss: 1.586..  Test Accuracy: 0.333
Epoch: 10/50.. Training Loss: 1.481..  Test Loss: 1.489..  Test Accuracy: 0.391
Epoch: 20/50.. Training Loss: 1.294..  Test Loss: 1.385..  Test Accuracy: 0.464
Epoch: 30/50.. Training Loss: 1.159..  Test Loss: 1.328..  Test Accuracy: 0.583
Epoch: 40/50.. Training Loss: 1.049..  Test Loss: 1.164..  Test Accuracy: 0.740
Epoch: 49/50.. Training Loss: 1.026..  Test Loss: 1.063..  Test Accuracy: 0.844
Epoch: 50/50.. Training Loss: 1.010..  Test Loss: 1.060..  Test Accuracy: 0.844
```


In [ ]:
# Save trained model
torch.save(model.state_dict(), 'emotion_model.pth')


### Model Architecture

- **Embedding layer**: Pretrained GloVe 50-dimensional word vectors (frozen)
- **LSTM**: 2-layer bidirectional LSTM with hidden size 128 and dropout 0.5
- **FC layer**: Maps LSTM output → 5 emotion classes
- **Output**: Softmax over 5 classes


Testing the Model Accuracy

In [ ]:
test_loss = 0
accuracy = 0
model.eval()
with torch.no_grad():
    for sentences, labels in test_loader:
        sentences, labels = sentences.to(device), labels.to(device)
        ps = model(sentences)
        test_loss += criterion(ps, labels).item()

        # Accuracy
        top_p, top_class = ps.topk(1, dim=1)
        equals = top_class == labels.view(*top_class.shape)
        accuracy += torch.mean(equals.type(torch.FloatTensor))
model.train()
print("Test Loss: {:.3f}.. ".format(test_loss/len(test_loader)),
      "Test Accuracy: {:.3f}".format(accuracy/len(test_loader)))
running_loss = 0

Testing the model with any sentence

In [ ]:
def predict(input_text, print_sentence=True):
  labels_dict = {
		0 : "❤️ Loving",
		1 : "⚽️ Playful",
		2 : "😄 Happy",
		3 : "😞 Annoyed",
		4 : "🍽 Foodie",
	}

  # Convert the input to the model
  x_test = np.array([input_text])
  X_test_indices = sentences_to_indices(x_test, word_to_index, maxLen)
  sentences = torch.tensor(X_test_indices).type(torch.LongTensor)

  # Get the class label
  ps = model(sentences)
  top_p, top_class = ps.topk(1, dim=1)
  label = int(top_class[0][0])

  if print_sentence:
    print("\nInput Text: \t"+ input_text +'\nEmotion: \t'+  labels_dict[label])

  return label